# Aspect 3 — Do Node Features Matter in Relational GNNs?

Before message passing, every node needs an *initial representation*. We fix a
**heterogeneous HGT** backbone and swap only the input encoder, comparing three
strategies along three axes: **downstream performance**, **model complexity
(#parameters)**, and **usability**.

| Strategy | Input | Encoder | Idea |
|----------|-------|---------|------|
| **ID** | learnable id | `Embedding[num_nodes → hidden]` per type | No tuple content — one embedding per node (structure-only). |
| **Column-wise** | encoded cells | `Linear[feat_dim → hidden]` per type | Each cell encoded by column type (numeric / categorical / datetime). |
| **LLM** | tuple string | `Linear[384 → hidden]` per type | `"col=val, …"` embedded with a frozen sentence-transformer. |

**Datasets** (different from aspect1): `rel-f1/driver-dnf` (small, full graph) and
`rel-event/user-repeat` (larger, non-target tables capped — same sample for all three).

All three variants share the identical backbone, hidden dim, depth, heads, optimizer,
loader and preprocessed graph; **only the input encoder changes.**

## 1. Setup

In [ ]:
# Set True to skip preprocessing + training and analyze the committed results/metrics.csv
RESULTS_ONLY = False

import json, os, subprocess, sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

_here = Path(os.path.abspath(""))
ROOT  = _here if _here.name == "aspect3" else _here / "aspect3"
os.chdir(ROOT)

PROCESSED   = ROOT / "processed"
CHECKPOINTS = ROOT / "checkpoints"
RESULTS_CSV = ROOT / "results" / "metrics.csv"
PYTHON      = sys.executable

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "| ROOT:", ROOT)

## 2. Datasets & Tasks

Two RelBench datasets with binary entity-classification targets, chosen to differ
from aspect1's rel-stack / rel-avito.

### rel-f1 / driver-dnf
**Source:** Formula 1 racing history.
**Task:** Predict whether a driver *does not finish* (DNF) their next race.
**Why:** Small (tens of thousands of rows) with healthy class balance, so the heavy
LLM encoding runs over the **full graph** — no sampling needed.

### rel-event / user-repeat
**Source:** Event-recommendation platform.
**Task:** Predict whether a user attends another event within 7 days.
**Why:** A larger, denser graph in a different domain. Non-target tables are capped
to 60k rows (uniform, seeded) to keep the LLM pass and the ID-embedding table
tractable; the **same sampled graph is used for all three feature strategies**.

In [ ]:
TASKS = [
    ("rel-f1",    "driver-dnf"),
    ("rel-event", "user-repeat"),
]

for dataset, task in TASKS:
    meta_path = PROCESSED / dataset / task / "meta.json"
    if not meta_path.exists():
        print(f"{dataset}/{task}: not preprocessed yet")
        continue
    meta = json.load(open(meta_path))
    print(f"\n{dataset}/{task}")
    print(f"  Target node : {meta['target_node']}")
    print(f"  Node types  : {len(meta['node_types'])}  ({', '.join(meta['node_types'])})")
    print(f"  Edge types  : {len(meta['edge_types'])}")
    print(f"  Column dims : {meta['node_feat_dims']}")
    print(f"  ID table sz : {meta['node_num_global']}  (sum={sum(meta['node_num_global'].values())})")
    print(f"  LLM dim     : {meta.get('llm_dim')}   sample cap: {meta.get('sample_cap')}")

## 3. Preprocessing

`preprocess.py` builds `train/val/test.pt` + `meta.json` per dataset. In a **single
pass** it stores all three inputs on every node so the strategies compare on the exact
same graph:

- `node.x` — column-wise features (StandardScaler / OrdinalEncoder / datetime offsets, fit on train)
- `node.gid` — stable global id (from the full db) → ID-embedding lookup
- `node.x_llm` — sentence-transformer (`all-MiniLM-L6-v2`) embedding of the tuple string

Temporal splits via `db.upto(cutoff)`; FK→PK edges plus reversed `rev_` edges.

In [ ]:
if RESULTS_ONLY:
    print("RESULTS_ONLY=True — skipping preprocessing.")
else:
    def done():
        return all((PROCESSED / d / t / "meta.json").exists() for d, t in TASKS)
    if done():
        print("Preprocessed data found — skipping.")
    else:
        print("Running preprocess.py (downloads + LLM encoding; slow on first run) …")
        r = subprocess.run([PYTHON, "preprocess.py"], cwd=ROOT)
        if r.returncode != 0:
            raise RuntimeError("preprocess.py failed — see output above.")

## 4. Model — HGT with three input encoders

A shared `HGTConv` backbone (`hidden=64`, 2 layers, 2 heads, dropout 0.3, sigmoid head).
Only the encoder mapping raw node inputs → `hidden` changes:

```
ID      : Embedding(num_nodes_type, 64)   indexed by node.gid     # params ∝ #nodes
Column  : Linear(feat_dim_type, 64)       on node.x
LLM     : Linear(384, 64)                 on node.x_llm
```

The cell below reports the parameter counts per strategy — the **model-complexity axis**.
The ID encoder dominates because it stores an embedding per node.

In [ ]:
import sys; sys.path.insert(0, str(ROOT))
from models import build_model, count_parameters, count_encoder_parameters
from train import load_meta, load_split

rows = []
for dataset, task in TASKS:
    if not (PROCESSED / dataset / task / "meta.json").exists():
        continue
    meta = load_meta(dataset, task)
    data = load_split(dataset, task, "train")
    for fm in ["id", "column", "llm"]:
        m = build_model(
            metadata=data.metadata(), target_node_type=meta["target_node"],
            feat_mode=fm, feat_dims=meta["node_feat_dims"],
            num_global=meta["node_num_global"], llm_dim=meta.get("llm_dim", 0),
            num_layers=2, hidden_dim=64, heads=2, dropout=0.3,
        )
        rows.append(dict(dataset=dataset, task=task, feat_mode=fm,
                         total_params=count_parameters(m),
                         encoder_params=count_encoder_parameters(m)))
params_df = pd.DataFrame(rows)
print(params_df.to_string(index=False)) if rows else print("Preprocess first to compute param counts.")

## 5. Experiment Configuration

**12 runs** = 2 datasets × 3 feature modes × 2 seeds.

| Axis | Values |
|------|--------|
| Dataset/Task | rel-f1/driver-dnf, rel-event/user-repeat |
| Model | HGT (hidden=64, 2 layers, 2 heads) |
| Feature mode | ID, Column-wise, LLM |
| Seed | 0, 1 |

Adam (lr=1e-3, wd=1e-5), ≤100 epochs, early stopping (patience 10) on val AUPRC,
batch 512 with NeighborLoader. Metrics: ROC AUC, AUPRC, precision, recall.

In [ ]:
FEAT_MODES = ["id", "column", "llm"]
NUM_LAYERS, SEEDS = 2, [0, 1]

def ckpt_path(dataset, task, feat_mode, seed):
    return CHECKPOINTS / dataset / task / f"hgt_{feat_mode}_L{NUM_LAYERS}_s{seed}.pt"

configs = [dict(dataset=d, task=t, feat_mode=fm, seed=s)
           for d, t in TASKS for fm in FEAT_MODES for s in SEEDS]
pending = [c for c in configs if not ckpt_path(c["dataset"], c["task"], c["feat_mode"], c["seed"]).exists()]
print(f"Total: {len(configs)}   Done: {len(configs)-len(pending)}   Pending: {len(pending)}")

## 6. Training

Runs all pending configs (completed ones are skipped). ~1–4 min per run on GPU.

In [ ]:
if RESULTS_ONLY:
    print("RESULTS_ONLY=True — skipping training.")
elif not pending:
    print("All runs complete — skipping training.")
else:
    print(f"Running {len(pending)} run(s) …")
    r = subprocess.run([PYTHON, "-u", "run_experiments.py"], cwd=ROOT)
    if r.returncode != 0:
        raise RuntimeError("run_experiments.py failed — see output above.")

## 7. Results

Load `results/metrics.csv` (one row per run) and analyze feature strategy along the
three axes: performance, parameters, usability.

In [ ]:
if not RESULTS_CSV.exists():
    raise FileNotFoundError(f"No results at {RESULTS_CSV}. Run training first.")
df = pd.read_csv(RESULTS_CSV)
if len(df) == 0:
    print("metrics.csv is empty — run training (or set RESULTS_ONLY=True after a cluster run).")
print(f"Loaded {len(df)} rows")
df.head()

In [ ]:
df = df.drop_duplicates(subset=["dataset","task","feat_mode","num_layers","seed"], keep="last").reset_index(drop=True)
FM_LABEL = {"id": "ID", "column": "Column-wise", "llm": "LLM"}
if len(df):
    df["Feat"]    = df["feat_mode"].map(FM_LABEL)
    df["Dataset"] = df["dataset"] + "/" + df["task"]
    agg = df.groupby(["Dataset","Feat"])[["test_auc","test_auprc","test_precision","test_recall"]].mean().round(4)
    print(agg.to_string())

### Figure 1 — Downstream performance by feature strategy

In [ ]:
if len(df):
    datasets = df["Dataset"].unique()
    order    = ["ID", "Column-wise", "LLM"]
    colors   = {"ID": "#B0779E", "Column-wise": "#4C72B0", "LLM": "#55A868"}
    metrics  = [("test_auprc", "Test AUPRC"), ("test_auc", "Test ROC AUC")]

    fig, axes = plt.subplots(len(datasets), 2, figsize=(12, 4.2*len(datasets)), squeeze=False)
    for r, dataset in enumerate(datasets):
        sub = df[df["Dataset"] == dataset]
        for c, (mcol, mlabel) in enumerate(metrics):
            ax = axes[r][c]
            means = [sub[sub["Feat"]==f][mcol].mean() for f in order]
            errs  = [sub[sub["Feat"]==f][mcol].std(ddof=0) for f in order]
            bars = ax.bar(order, means, yerr=errs, capsize=4,
                          color=[colors[f] for f in order], edgecolor="white")
            for b, v in zip(bars, means):
                if not np.isnan(v):
                    ax.text(b.get_x()+b.get_width()/2, v, f"{v:.3f}",
                            ha="center", va="bottom", fontsize=9)
            ax.set_title(f"{dataset} — {mlabel}", fontsize=10, fontweight="bold")
            ax.set_ylabel(mlabel); ax.set_ylim(0, 1)
            ax.spines[["top","right"]].set_visible(False)
    fig.suptitle("Figure 1: Performance by Node-Feature Strategy (mean ± std over seeds)",
                 fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.savefig(ROOT/"results"/"fig1_performance.png", dpi=120, bbox_inches="tight"); plt.show()

### Figure 2 — Model complexity (parameter count)

Total trainable parameters per strategy, log scale. The ID embedding table makes that variant's parameter count scale with the number of nodes.

In [ ]:
if len(df):
    fig, axes = plt.subplots(1, len(datasets), figsize=(6*len(datasets), 4.2), squeeze=False)
    for c, dataset in enumerate(datasets):
        ax = axes[0][c]
        sub = df[df["Dataset"] == dataset]
        vals = [sub[sub["Feat"]==f]["num_params"].mean() for f in order]
        bars = ax.bar(order, vals, color=[colors[f] for f in order], edgecolor="white")
        for b, v in zip(bars, vals):
            if not np.isnan(v):
                ax.text(b.get_x()+b.get_width()/2, v, f"{int(v):,}",
                        ha="center", va="bottom", fontsize=8)
        ax.set_yscale("log"); ax.set_ylabel("Trainable parameters (log)")
        ax.set_title(dataset, fontsize=10, fontweight="bold")
        ax.spines[["top","right"]].set_visible(False)
    fig.suptitle("Figure 2: Model Complexity by Feature Strategy", fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.savefig(ROOT/"results"/"fig2_params.png", dpi=120, bbox_inches="tight"); plt.show()

### Figure 3 — Performance vs. complexity

Test AUPRC against total parameters (log-x). Upper-left is the sweet spot: strong performance at low parameter cost.

In [ ]:
if len(df):
    fig, ax = plt.subplots(figsize=(7.5, 5))
    markers = {"rel-f1/driver-dnf": "o", "rel-event/user-repeat": "s"}
    for dataset in datasets:
        sub = df[df["Dataset"] == dataset]
        g = sub.groupby("Feat").agg(p=("num_params","mean"), a=("test_auprc","mean")).reindex(order)
        for f in order:
            if f in g.index and not np.isnan(g.loc[f,"a"]):
                ax.scatter(g.loc[f,"p"], g.loc[f,"a"], s=140, color=colors[f],
                           marker=markers.get(dataset,"o"), edgecolor="black", zorder=3)
                ax.annotate(f, (g.loc[f,"p"], g.loc[f,"a"]), textcoords="offset points",
                            xytext=(6,4), fontsize=9)
    ax.set_xscale("log"); ax.set_xlabel("Trainable parameters (log)"); ax.set_ylabel("Test AUPRC")
    ax.set_title("Figure 3: Performance vs. Complexity", fontsize=12, fontweight="bold")
    handles = [plt.Line2D([0],[0], marker=markers[d], color="gray", linestyle="",
               markersize=9, label=d) for d in datasets if d in markers]
    ax.legend(handles=handles, fontsize=8)
    ax.spines[["top","right"]].set_visible(False)
    plt.tight_layout(); plt.savefig(ROOT/"results"/"fig3_tradeoff.png", dpi=120, bbox_inches="tight"); plt.show()

### Table 1 — Full results summary

In [ ]:
if len(df):
    summary = (df.groupby(["Dataset","Feat"])
                 [["val_auprc","test_auprc","test_auc","test_precision","test_recall","num_params","train_time_sec"]]
                 .mean().round(4))
    summary.columns = ["Val AUPRC","Test AUPRC","Test AUC","Test Prec","Test Recall","Params","Train s"]
    print(summary.to_string())

### Table 2 — Usability (qualitative axis)

The third axis the assignment asks about. Not measured numerically; assessed from the
implementation effort and operational constraints of each strategy.

In [ ]:
usability = pd.DataFrame([
    dict(Strategy="ID", Implementation="Trivial (embedding table)",
         Preprocessing="None (just an id)", Cost="Low compute, high memory (∝ #nodes)",
         Inductive="No — new/test nodes get untrained embeddings"),
    dict(Strategy="Column-wise", Implementation="Moderate (type-aware encoders + scaling)",
         Preprocessing="Fit encoders on train", Cost="Low",
         Inductive="Yes"),
    dict(Strategy="LLM", Implementation="Easy to call, heavy to run",
         Preprocessing="Stringify + embed every tuple (GPU)", Cost="High (one-off embedding pass)",
         Inductive="Yes (frozen encoder generalizes to new tuples)"),
])
print(usability.to_string(index=False))

## 8. Analysis & Conclusions

### What we tested
Three initial-representation strategies for an HGT model — **ID** (learnable per-node
embedding, no content), **Column-wise** (typed cell encoders), and **LLM** (frozen
sentence-transformer over the tuple string) — on two RelBench entity-classification
tasks, along three axes: performance, parameter count, and usability.

### Expectations (pre-registered)
- **ID** should trail on test: its embeddings are transductive, so entities seen only
  at test time carry no learned signal, and it exploits only graph structure.
- **Column-wise** should be a strong, cheap baseline — the model sees real tuple content
  at low parameter cost.
- **LLM** should match or beat column-wise when free-text / high-cardinality categorical
  content carries signal, at a large one-off embedding cost but a *small* trainable
  encoder (`Linear[384→64]`), and it is naturally inductive.
- On **parameters**, ID ≫ LLM ≈ Column-wise (the ID embedding table dominates).

### Findings
*(Fill in after the cluster run populates `results/metrics.csv`; reference Figures 1–3
and Tables 1–2.)*
- **Performance (Fig 1):** …
- **Complexity (Fig 2):** …
- **Trade-off (Fig 3):** which strategy sits upper-left (best AUPRC per parameter)?
- **Usability (Table 2):** ID is simplest to code but doesn't generalize; column-wise
  is the best effort/reward; LLM is a one-line call but a heavy, one-off preprocessing step.

### Limitations & future work
- Non-target tables in rel-event are uniformly subsampled (60k cap) for LLM feasibility;
  this thins connectivity equally for all three variants (fair comparison) but is an
  approximation of the full task.
- One LLM (`all-MiniLM-L6-v2`); larger or domain-tuned encoders may change the LLM row.
- ID is transductive by construction — an inductive id scheme (e.g. hashing) is future work.